# CLCT threshold selection (Youden's J), with parallax and detectability corrections

A focused re-implementation of section 5 (method 4c) of `CLCT_binary_verification.ipynb`: sweep a CLCT
threshold, pick `tau*` maximising Youden's J = POD - POFD on calibration data, and report skill on
evaluation data that took no part in the choice. Everything else from that notebook is dropped so that the
threshold itself can be done properly.

The original quoted `tau*` as a single number from a single split against an uncorrected satellite mask.
Four things limit how much that number means, and each is addressed here:

1. **Parallax.** MSG sees Switzerland at ~54 deg viewing zenith, so a cloud top at height `z` is reported
   about `1.38 z` away from the ground point it actually sits over -- 3-4 grid cells for high cloud. The
   mask is displaced back onto true ground positions using the `ctth_alti` retrieval in the same file
   (section 4). Clear pixels are displaced too, by terrain height, which is not negligible in the Alps.
2. **Retrieval quality and cloud class.** These files carry **no** `cma_quality` / `cma_conditions`
   variables -- only the CTTH product has flags. What is available (`ctth_quality`, `ctth_conditions`,
   `ctth_status_flag`, `ctth_method`) qualifies the *height*, not the cloud bit, so it gates the parallax
   correction rather than the mask. Screening of the mask itself goes through the `ct` cloud-type product
   instead. Section 3 states exactly what this can and cannot do.
3. **Scan timing.** SEVIRI scans south to north across a 12m23s acquisition, so the Alps are imaged about
   10 minutes after the nominal slot time stamped on the filename, while ICON `lff` is instantaneous on the
   hour. Pairing hour `T` with slot `T` is therefore a ~10 minute mismatch; the `T-15min` slot is closer.
   Section 5 derives this from the files' own `start_time`/`end_time` and picks the better slot.
4. **Detectability.** MSG cannot see what is optically too thin, and NWCSAF says so itself through the
   `Fractional_clouds` / `High_semitransparent_thin` cloud types and the
   `Too_thin_clouds_no_reliable_method` status bit. ICON's CLCT counts that cloud regardless. Section 6
   adds the complementary model-side view: a CLCT recomputed with optically undetectable layers removed.

`tau*` is then reported not as a point value but with a moving-block bootstrap interval and k-fold
cross-validation over time blocks (sections 11-12), because hourly steps are not independent samples and
the J curve is typically flat near its maximum. Section 10 is the point of the whole notebook: the same
threshold refitted under each correction in turn, so the movement in `tau*` is attributable.

**Data actually on disk** (verified, not assumed): NWCSAF 15-min slots cover 2025-10-04 00:00 ..
2025-10-10 00:00 with exactly two missing (2025-10-06 12:00 and 12:15); experiment 801/802 `lff` covers
2025-10-04 01:00 .. 2025-10-10 01:00. The NWCSAF grid is a regular 341x482 lat/lon box at 0.0388 x 0.0270
deg (~2.97 x 3.00 km) spanning 1.08W..17.58E, 41.91N..51.09N, with latitude **descending**. `cma` is a
clean 0/1 field with no fill value in these exports.

In [ ]:
import os
os.environ["ECCODES_VERSION_CHECK_OFF"] = "1"

from datetime import datetime, timedelta

import numpy as np
import xarray as xr
import shapely
import earthkit.data as ekd

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

if shapely.__version__ < "2":
    raise RuntimeError(f"needs shapely >= 2, found {shapely.__version__}")

# =============================================================================
# Config
# =============================================================================
EXP = "801"
NWCSAF_DIR = "/scratch/mch/jdelbeke/nwcsaf"
ICON_DIR = f"/store_new/mch/msopr/jdelbeke/ICON_TST/{EXP}/FG25/det"
ICON_GRID_FILE = "/oprusers/osm/opr.inn/data/ICON_INPUT/ICON-CH1-EPS/icon_grid_0001_R19B08_mch.nc"

CACHE_DIR = "/scratch/mch/jdelbeke"
WEIGHTS_CACHE = f"{CACHE_DIR}/regrid_weights_icon_nwcsaf.npz"
HIST_CACHE = f"{CACHE_DIR}/clct_threshold_hists_{EXP}.npz"
DETECT_CACHE = f"{CACHE_DIR}/clct_detectable_{EXP}.npz"


def nwcsaf_file(t):
    return f"{NWCSAF_DIR}/MSG_nwcsaf_cosmo1eqc3km_{t:%Y%m%d%H%M}.nc"


def icon_file(t):
    return f"{ICON_DIR}/lff{t:%Y%m%d%H}"


# NWCSAF runs out at 2025-10-10 00:00, so the last usable ICON hour is that one
# (and with scan-time matching it pairs with the 2025-10-09 23:45 slot).
START = datetime(2025, 10, 4, 1)
N_HOURS = 144
TIMES = [START + timedelta(hours=h) for h in range(N_HOURS)]
SPLIT_DATE = datetime(2025, 10, 7)     # daytime from here on = evaluation pool

DOMAIN = None            # None = whole NWCSAF grid, or [lonW, lonE, latS, latN]
MIN_COVERAGE = 0.999     # target cells less covered by ICON than this -> NaN
CHUNK = 200_000

# --- threshold grid ---------------------------------------------------------
# CLCT is 0..100. Binning it lets every J evaluation below (bootstrap,
# cross-validation, every stratum) be a sum over precomputed per-timestep
# histograms rather than a fresh pass over the pixels.
#
# The binning is exact, not approximate: bins are floor(CLCT * 10) and the
# threshold is lower-inclusive, so "CLCT >= TAUS[k]" is *identically* the same
# set of pixels as "bin >= k". Rounding to the nearest bin instead would make
# the histogram disagree with a direct contingency count by a few times 1e-3
# near every threshold. Section 9 asserts the identity rather than trusting it.
BIN_PER_PCT = 10
NBIN = 100 * BIN_PER_PCT + 1
TAUS = np.arange(NBIN) / BIN_PER_PCT

# --- corrections ------------------------------------------------------------
PARALLAX = True          # displace the mask onto true ground positions
PLX_STRICT_QUALITY = False  # also drop ctth heights flagged 'bad' (see section 4)
SCAN_TIME_MATCH = True   # pair each ICON hour with the slot imaged closest to it
SAT_LON = 0.0            # Meteosat-10 sub-satellite longitude
DETECTABILITY_PASS = True   # model-side optical-depth filter (SLOW, cached)
TAU_DETECT = 0.2            # optical depth a model layer must exceed to count

# Elevation bands for the terrain stratification (m).
ELEV_EDGES = [-100, 800, 1500, 5000]

# Statistics
DECORR_H = 4             # block length for the bootstrap; re-measured in sec. 8
N_BOOT = 1000
N_FOLDS = 6
MIN_EFFECTIVE_N = 30

print(f"exp {EXP}: {len(TIMES)} hourly steps, {TIMES[0]} .. {TIMES[-1]}")

## 1. Conservative regridding (ICON native mesh -> NWCSAF grid)

Unchanged from `CLCT_binary_verification.ipynb`, and the weights cache is shared with it: every ICON
triangle is clipped against every NWCSAF cell it overlaps and weighted by the exact overlap area, so
`remap()` is two `bincount` calls. `check_partition_of_unity()` re-validates on every run, cache or not.

The one thing worth restating: the target grid's latitude axis **descends**, so `dlat` is negative
everywhere below. Absolute values are used for cell sizes and the signed value for index arithmetic --
getting that backwards would flip the parallax displacement north/south, which is exactly the error this
notebook is trying to remove.

In [ ]:
def read_grid(path, domain=None):
    """1-D coordinates of the NWCSAF grid, validated as regular."""
    with xr.open_dataset(path) as ds:
        lon2 = np.asarray(ds["longitude"].values)
        lat2 = np.asarray(ds["latitude"].values)

    if lon2.ndim == 2:
        if (np.abs(lon2 - lon2[0, :]).max() > 1e-9
                or np.abs(lat2 - lat2[:, [0]]).max() > 1e-9):
            raise ValueError("2-D coordinates are not a regular lat/lon grid")
        lon, lat = lon2[0, :], lat2[:, 0]
    else:
        lon, lat = lon2, lat2

    dlon = float(np.median(np.diff(lon)))
    dlat = float(np.median(np.diff(lat)))     # negative: latitude descends
    if (np.abs(np.diff(lon) - dlon).max() > 1e-9
            or np.abs(np.diff(lat) - dlat).max() > 1e-9):
        raise ValueError("grid spacing is not uniform")

    if domain is not None:
        jx = np.where((lon >= domain[0]) & (lon <= domain[1]))[0]
        iy = np.where((lat >= domain[2]) & (lat <= domain[3]))[0]
        lon, lat = lon[jx], lat[iy]
    return lon, lat, dlon, dlat


def read_icon_cells(grid_file, n_expected, lon_c, lat_c):
    """ICON triangle vertices in degrees, validated against the field."""
    with xr.open_dataset(grid_file) as g:
        vlon = np.asarray(g["clon_vertices"].values, dtype="float64")
        vlat = np.asarray(g["clat_vertices"].values, dtype="float64")
    if vlon.shape[0] != n_expected:
        raise ValueError(f"grid file has {vlon.shape[0]} cells, field has {n_expected}")
    if np.abs(vlat).max() <= np.pi + 1e-6:      # no units attribute; detect radians
        vlon, vlat = np.rad2deg(vlon), np.rad2deg(vlat)
    dv = max(np.abs(vlon.mean(axis=1) - lon_c).max(),
             np.abs(vlat.mean(axis=1) - lat_c).max())
    if dv > 1e-3:
        raise ValueError("vertices do not match the field's cell centres")
    return vlon, vlat


def triangle_polygons(vlon, vlat, sub):
    rings = np.stack([vlon[sub], vlat[sub]], axis=-1)
    rings = np.concatenate([rings, rings[:, :1, :]], axis=1)
    return shapely.polygons(rings)


def build_weights(lon, lat, dlon, dlat, vlon, vlat):
    ny, nx = lat.size, lon.size
    ncell = ny * nx
    hx, hy = abs(dlon) / 2, abs(dlat) / 2

    near = ((vlon.max(axis=1) >= lon.min() - hx)
            & (vlon.min(axis=1) <= lon.max() + hx)
            & (vlat.max(axis=1) >= lat.min() - hy)
            & (vlat.min(axis=1) <= lat.max() + hy))
    sub = np.where(near)[0]
    tri = triangle_polygons(vlon, vlat, sub)

    LON, LAT = np.meshgrid(lon, lat)
    cells = shapely.box((LON - hx).ravel(), (LAT - hy).ravel(),
                        (LON + hx).ravel(), (LAT + hy).ravel())

    ti, ci = shapely.STRtree(cells).query(tri, predicate="intersects")
    w = np.empty(ti.size)
    for a in range(0, ti.size, CHUNK):
        b = min(a + CHUNK, ti.size)
        w[a:b] = shapely.area(shapely.intersection(tri[ti[a:b]], cells[ci[a:b]]))
    keep = w > 0
    ti, ci, w = ti[keep], ci[keep], w[keep]

    coverage = np.bincount(ci, weights=w, minlength=ncell) / (abs(dlon) * abs(dlat))
    print(f"target cells >= {MIN_COVERAGE} covered: {(coverage >= MIN_COVERAGE).sum()}/{ncell}")
    return sub, ti, ci, w, coverage.reshape(ny, nx)


def check_partition_of_unity(sub, ti, w, vlon, vlat, lon, lat, dlon, dlat, tol=1e-9):
    """Each interior triangle's area must be exactly partitioned among the cells
    it overlaps. Restricted to triangles fully inside the target grid -- edge
    triangles legitimately lose area, and triangles selected only by their
    bounding box contribute none, so both would pass trivially."""
    hx, hy = abs(dlon) / 2, abs(dlat) / 2
    want = shapely.area(triangle_polygons(vlon, vlat, sub))
    got = np.bincount(ti, weights=w, minlength=sub.size)
    interior = ((vlon[sub].min(axis=1) > lon.min() - hx)
                & (vlon[sub].max(axis=1) < lon.max() + hx)
                & (vlat[sub].min(axis=1) > lat.min() - hy)
                & (vlat[sub].max(axis=1) < lat.max() + hy))
    err = float(np.abs(got[interior] / want[interior] - 1).max())
    print(f"partition of unity ({int(interior.sum())} interior triangles of {sub.size} "
          f"selected): max |sum(overlap)/area - 1| = {err:.2e}")
    if err > tol:
        raise RuntimeError("regridding weights do not partition triangle area")
    return err


def remap(values, sub, ti, ci, w, coverage, shape):
    v = np.asarray(values, dtype="float64").ravel()[sub]
    ok = np.isfinite(v[ti])
    ncell = shape[0] * shape[1]
    num = np.bincount(ci[ok], weights=w[ok] * v[ti][ok], minlength=ncell)
    den = np.bincount(ci[ok], weights=w[ok], minlength=ncell)
    out = np.full(ncell, np.nan)
    good = (coverage.ravel() >= MIN_COVERAGE) & (den > 0)
    out[good] = num[good] / den[good]
    return out.reshape(shape)

In [ ]:
# --- grids and weights ------------------------------------------------------
probe_time = TIMES[0]
lon, lat, dlon, dlat = read_grid(nwcsaf_file(probe_time), DOMAIN)
shape = (lat.size, lon.size)
CELL_KM_X = abs(dlon) * 111.32 * np.cos(np.deg2rad(float(np.mean(lat))))
CELL_KM_Y = abs(dlat) * 111.32
print(f"target grid {shape}, cell {CELL_KM_X:.2f} x {CELL_KM_Y:.2f} km, "
      f"dlon {dlon:+.5f}, dlat {dlat:+.5f} "
      f"(latitude {'descends' if dlat < 0 else 'ascends'})")

fl = ekd.from_source("file", icon_file(probe_time)).to_fieldlist()
xa = fl.sel({"parameter.variable": "CLCT"})[0].to_xarray()
name0 = "CLCT" if "CLCT" in xa else list(xa.data_vars)[0]
lon_c = np.asarray(xa["longitude"].values).ravel()
lat_c = np.asarray(xa["latitude"].values).ravel()
vlon, vlat = read_icon_cells(ICON_GRID_FILE, lon_c.size, lon_c, lat_c)

if WEIGHTS_CACHE and os.path.exists(WEIGHTS_CACHE):
    z = np.load(WEIGHTS_CACHE)
    sub, ti, ci, w, coverage = z["sub"], z["ti"], z["ci"], z["w"], z["coverage"]
    print(f"weights loaded from {WEIGHTS_CACHE}")
else:
    sub, ti, ci, w, coverage = build_weights(lon, lat, dlon, dlat, vlon, vlat)
    if WEIGHTS_CACHE:
        np.savez_compressed(WEIGHTS_CACHE, sub=sub, ti=ti, ci=ci, w=w, coverage=coverage)
        print(f"weights saved to {WEIGHTS_CACHE}")

check_partition_of_unity(sub, ti, w, vlon, vlat, lon, lat, dlon, dlat)
in_domain = coverage >= MIN_COVERAGE

# --- static model orography, on the target grid ------------------------------
hs = fl.sel({"parameter.variable": "HSURF"})
if len(hs) == 0:
    raise RuntimeError("HSURF not in the lff file -- needed for terrain parallax "
                       "and the elevation stratification")
xh = hs[0].to_xarray()
nmh = "HSURF" if "HSURF" in xh else list(xh.data_vars)[0]
HSURF = remap(np.asarray(xh[nmh].values, dtype="float64").ravel(),
              sub, ti, ci, w, coverage, shape)
HSURF_KM = np.where(np.isfinite(HSURF), HSURF, 0.0) / 1000.0

ELEV_BAND = np.full(shape, -1, dtype=int)
elev_names = []
for k in range(len(ELEV_EDGES) - 1):
    lo_e, hi_e = ELEV_EDGES[k], ELEV_EDGES[k + 1]
    ELEV_BAND[np.isfinite(HSURF) & (HSURF >= lo_e) & (HSURF < hi_e)] = k
    elev_names.append(f"{lo_e}-{hi_e} m")

print(f"\n{int(in_domain.sum())} cells inside the ICON domain")
print("elevation bands (cells inside the domain):")
for k, nm in enumerate(elev_names):
    print(f"  {nm:12s} {int(((ELEV_BAND == k) & in_domain).sum()):6d}")

## 2. Viewing geometry: parallax displacement and scan timing

Both corrections come out of the same geostationary geometry, so they are derived together and per pixel
rather than for the domain centre only.

**Displacement.** For a pixel at `(lat, lon)` seen from a satellite at `SAT_LON`, the viewing zenith angle
follows from the angular distance `psi` to the sub-satellite point. A feature at height `z` is reported
`tan(vza) * z` further from the sub-satellite point than it really is, so the correction moves it
`tan(vza) * z` *towards* the sub-satellite point, along the great-circle bearing to it. Over this domain
`tan(vza)` is around 1.4, and the direction is essentially due south with a small westward component.

**Scan timing.** SEVIRI acquires the full disc south to north. The files themselves carry `start_time` and
`end_time` on every variable (12m23s apart), so the acquisition window is read rather than assumed; only
the *fraction* of it at which this domain is imaged is computed, from the north-south geostationary angle
`y` and the disc half-extent. Section 5 checks the result against those attributes.

In [ ]:
R_E, R_S = 6371.0, 42164.0          # km: Earth radius, geostationary orbit radius
Y_DISC = 0.1518                     # rad: half the north-south angular extent of the disc

LON2, LAT2 = np.meshgrid(lon, lat)
_la, _dlo = np.deg2rad(LAT2), np.deg2rad(LON2 - SAT_LON)

PSI = np.arccos(np.cos(_la) * np.cos(_dlo))                 # angular distance to sub-sat point
VZA = np.arctan2(R_S * np.sin(PSI), R_S * np.cos(PSI) - R_E)
# initial great-circle bearing from each pixel towards the sub-satellite point
BRG = np.arctan2(np.sin(-_dlo), -np.sin(_la) * np.cos(-_dlo))

# displacement per km of feature height, in degrees, towards the sub-satellite point
DLAT_PER_KM = np.tan(VZA) * np.cos(BRG) / 111.32
DLON_PER_KM = np.tan(VZA) * np.sin(BRG) / (111.32 * np.cos(_la))

_c = (shape[0] // 2, shape[1] // 2)
print(f"domain centre {LAT2[_c]:.2f}N {LON2[_c]:.2f}E: viewing zenith "
      f"{np.rad2deg(VZA[_c]):.1f} deg, displacement {np.tan(VZA[_c]):.2f} x height")
print(f"viewing zenith across the domain: {np.rad2deg(VZA).min():.1f} .. "
      f"{np.rad2deg(VZA).max():.1f} deg")
print("\nshift applied to the observation, at the domain centre:")
for z in (0.5, 1, 2, 5, 8):
    print(f"  height {z:4.1f} km -> {z * DLAT_PER_KM[_c] / dlat:+5.2f} rows "
          f"({z * DLAT_PER_KM[_c] * 111.32:+5.1f} km north), "
          f"{z * DLON_PER_KM[_c] / dlon:+5.2f} cols")

# --- scan fraction: where in the south-to-north sweep this domain is imaged ---
def geos_y_angle(lat_deg, lon_deg, sat_lon=SAT_LON):
    """North-south geostationary viewing angle. Positive south, negative north."""
    c_lat = np.arctan(0.993243 * np.tan(np.deg2rad(lat_deg)))
    r_l = 6356.5838 / np.sqrt(1 - 0.00669438 * np.cos(c_lat) ** 2)
    dl = np.deg2rad(lon_deg - sat_lon)
    r1 = 42164.0 - r_l * np.cos(c_lat) * np.cos(dl)
    r2 = -r_l * np.cos(c_lat) * np.sin(dl)
    r3 = r_l * np.sin(c_lat)
    rn = np.sqrt(r1 ** 2 + r2 ** 2 + r3 ** 2)
    return np.arcsin(-r3 / rn)


lat0, lon0 = float(np.mean(lat)), float(np.mean(lon))
y_ang = geos_y_angle(lat0, lon0)
SCAN_FRAC = float((Y_DISC - y_ang) / (2 * Y_DISC))   # 0 at the south edge, 1 at the north
print(f"\ndomain centre is imaged {100 * SCAN_FRAC:.0f}% of the way through the "
      f"south-to-north sweep")

## 3. What the NWCSAF files actually contain

Worth being explicit, because it constrains what the quality screening can be. These exports are
satpy-resampled onto the COSMO-1 3 km grid and carry:

`cma`, `ct`, `ctth_alti`, `ctth_pres`, `ctth_tempe`, `ctth_quality`, `ctth_conditions`,
`ctth_status_flag`, `ctth_method`.

**There is no `cma_quality`, `cma_conditions` or `cma_status_flag`.** The cloud mask arrives as a bare 0/1
field with no confidence attached and no fill value. So "screen the mask by its quality flags" is not
available as such, and the two things that *are* available are used instead:

- **`ctth_*` flags gate the parallax correction, not the mask.** They describe the height retrieval, so
  they decide whether a cloudy pixel can be displaced reliably -- not whether it is really cloudy.
  `ctth_quality` resolves to good / questionable / bad through `(v & 56)`; `ctth_conditions` carries the
  illumination (`v & 6`: night / day / twilight), land/sea and sun-glint bits; `ctth_method` marks pixels
  with no reliable method at all.
- **`ct` screens the mask.** The cloud-type product separates the cases where a binary mask is least
  trustworthy: `Snow_over_land` and `Sea_ice` (bright surfaces misread as cloud, or cloud lost against
  them), `Fractional_clouds` (sub-pixel cloud, where a 0/1 answer is a coin flip), and
  `High_semitransparent_thin_clouds` (optically thin cirrus). These are stratified and optionally excluded
  in sections 10 and 13.

The illumination bits also replace the original notebook's hand-rolled solar zenith angle: the day / night
/ twilight split is stated by the product rather than recomputed, and twilight -- when the mask is at its
worst -- becomes its own category instead of being lumped into one side of a 90 degree cut.

In [ ]:
def decode_flags(da):
    """Named boolean layers from a CF flag variable, using the file's own
    flag_values/flag_mask attributes rather than a hard-coded bit layout.
    Duplicate flag_meanings (the files contain two 'not_used') get suffixed."""
    a = np.asarray(da.values)
    ai = np.where(np.isfinite(a), a, 0).astype("int64")
    meanings = da.attrs["flag_meanings"].split()
    values = np.asarray(da.attrs["flag_values"]).astype("int64")
    masks = da.attrs.get("flag_mask", da.attrs.get("flag_masks", values))
    masks = np.asarray(masks).astype("int64")
    out, seen = {}, {}
    for m, v, mk in zip(meanings, values, masks):
        seen[m] = seen.get(m, 0) + 1
        key = m if seen[m] == 1 else f"{m}__{seen[m]}"
        out[key] = (ai & int(mk)) == int(v)
    return out


CT_SNOW_ICE = (3, 4)          # Snow_over_land, Sea_ice
CT_FRACTIONAL = (10,)         # Fractional_clouds -- sub-pixel
CT_THIN = (11,)               # High_semitransparent_thin_clouds
CT_SEMITRANSP = (11, 12, 13, 14, 15)


def read_nwcsaf(path, domain=None):
    """Cloud mask plus everything needed to correct and screen it."""
    with xr.open_dataset(path) as ds:
        if domain is not None:
            lo1 = np.asarray(ds["longitude"].values)
            la1 = np.asarray(ds["latitude"].values)
            lo1 = lo1[0, :] if lo1.ndim == 2 else lo1
            la1 = la1[:, 0] if la1.ndim == 2 else la1
            jx = np.where((lo1 >= domain[0]) & (lo1 <= domain[1]))[0]
            iy = np.where((la1 >= domain[2]) & (la1 <= domain[3]))[0]
            sl = np.ix_(iy, jx)
        else:
            sl = (slice(None), slice(None))

        cma = np.asarray(ds["cma"].squeeze().values, dtype="float64")[sl]
        cma[cma == 255] = np.nan                       # no-op in these exports; kept defensively
        ct = np.asarray(ds["ct"].squeeze().values, dtype="float64")[sl]
        alti = np.asarray(ds["ctth_alti"].squeeze().values, dtype="float64")[sl]
        lo, hi = ds["ctth_alti"].attrs.get("valid_range", (-2000.0, 25000.0))
        alti[(alti < lo) | (alti > hi)] = np.nan

        qual = decode_flags(ds["ctth_quality"])
        cond = decode_flags(ds["ctth_conditions"])
        meth = decode_flags(ds["ctth_method"])
        stat = decode_flags(ds["ctth_status_flag"])
        qual = {k: v[sl] for k, v in qual.items()}
        cond = {k: v[sl] for k, v in cond.items()}
        meth = {k: v[sl] for k, v in meth.items()}
        stat = {k: v[sl] for k, v in stat.items()}

        t0 = str(ds["cma"].attrs.get("start_time", ""))
        t1 = str(ds["cma"].attrs.get("end_time", ""))

    return dict(cma=cma, ct=ct, alti=alti, qual=qual, cond=cond, meth=meth,
                stat=stat, start_time=t0, end_time=t1)


# --- inventory on one slot ---------------------------------------------------
S = read_nwcsaf(nwcsaf_file(datetime(2025, 10, 6, 11)), DOMAIN)
n = S["cma"].size
print(f"acquisition window: {S['start_time']} .. {S['end_time']}")
print(f"cma: {int(np.nansum(S['cma'] == 1))} cloudy, "
      f"{int(np.nansum(S['cma'] == 0))} clear, {int(np.isnan(S['cma']).sum())} missing")

cloudy = S["cma"] == 1
have_h = cloudy & np.isfinite(S["alti"])
print(f"cloudy pixels with a usable ctth_alti: {int(have_h.sum())}/{int(cloudy.sum())} "
      f"({100 * have_h.sum() / max(cloudy.sum(), 1):.1f}%) -- the rest cannot be "
      f"parallax-corrected")

for label, d in (("ctth_quality", S["qual"]), ("ctth_conditions", S["cond"]),
                 ("ctth_status_flag", S["stat"])):
    print(f"\n{label}:")
    for k, v in d.items():
        f = v.sum() / n
        if f > 0:
            print(f"    {k:48s} {100 * f:5.1f}%")

print("\nct cloud types present:")
ct_names = {1: "Cloud-free_land", 2: "Cloud-free_sea", 3: "Snow_over_land", 4: "Sea_ice",
            5: "Very_low", 6: "Low", 7: "Mid-level", 8: "High_opaque",
            9: "Very_high_opaque", 10: "Fractional", 11: "Hi_semitransp_thin",
            12: "Hi_semitransp_moderate", 13: "Hi_semitransp_thick",
            14: "Hi_semitransp_above_low_med", 15: "Hi_semitransp_above_snow_ice"}
for v_, c_ in zip(*np.unique(S["ct"][np.isfinite(S["ct"])], return_counts=True)):
    print(f"    {int(v_):2d} {ct_names.get(int(v_), '?'):30s} {100 * c_ / n:5.1f}%")

## 4. Parallax correction of the mask

Each pixel is moved from where MSG reports it to the ground point it actually sits over, and the corrected
mask is rebuilt by scattering pixels into the cells they land in:

- **Cloudy pixels with a usable `ctth_alti`** move by `tan(vza) * z` towards the sub-satellite point.
- **Clear pixels** move by `tan(vza) * HSURF`. This is not a refinement to be skipped: at 2.5 km of Alpine
  terrain the displacement is around 3.5 km, a full grid cell, and it is systematic rather than random.
- **Cloudy pixels without a height retrieval** (about 9% of cloudy pixels, `ctth_method` = no reliable
  method) cannot be placed at all and become NaN. Guessing a height for them would manufacture exactly the
  signal being measured.

Two things then happen that a "shift the field" implementation hides, and both are reported per timestep:

- **Gaps.** A ground cell can end up receiving no pixel at all -- it was hidden behind a displaced cloud.
  It is genuinely unobserved and stays NaN rather than being filled with the nearest value.
- **Collisions.** A cell can receive both a displaced cloudy and a displaced clear pixel. Cloud wins: if
  any line of sight over that ground point held cloud, the column is cloudy.

`ctth_quality` gates the correction rather than the mask -- a `bad` height would displace a real cloud to
the wrong place, which is worse than not moving it. Those pixels are treated like the ones with no height
at all. This is a deliberately conservative choice and it costs sample size; section 10 shows how much.

In [ ]:
def parallax_correct(sat, hsurf_km, strict_quality=PLX_STRICT_QUALITY):
    """Displace the mask onto true ground positions. Returns the corrected mask
    and a diagnostics dict.

    `strict_quality` also discards heights flagged `bad` by ctth_quality. It is
    off by default, and the reason is worth stating: not displacing a cloudy
    pixel is not a neutral act, it is equivalent to asserting the cloud top is
    at height zero. For a pixel the mask calls cloudy that is the one height it
    certainly is not. A `bad` retrieval is a poor estimate; no retrieval is a
    guaranteed-wrong one. In these files `bad` covers 28% of pixels, so strict
    gating leaves about a third of the domain unobserved -- the ladder in
    section 10 carries both variants so the trade is visible rather than
    assumed."""
    cma = sat["cma"]
    ny, nx = cma.shape
    cloudy = cma == 1
    clear = cma == 0

    h_km = sat["alti"] / 1000.0
    ok_h = np.isfinite(h_km)
    ok_h &= ~sat["meth"].get("No_reliable_method", np.zeros_like(ok_h))
    if strict_quality:
        ok_h &= ~sat["qual"].get("bad", np.zeros_like(ok_h))

    movable = cloudy & ok_h
    lost = cloudy & ~ok_h

    def scatter(mask, z_km):
        i0, j0 = np.nonzero(mask)
        lat_t = LAT2[i0, j0] + DLAT_PER_KM[i0, j0] * z_km[i0, j0]
        lon_t = LON2[i0, j0] + DLON_PER_KM[i0, j0] * z_km[i0, j0]
        i1 = np.rint((lat_t - lat[0]) / dlat).astype(int)
        j1 = np.rint((lon_t - lon[0]) / dlon).astype(int)
        keep = (i1 >= 0) & (i1 < ny) & (j1 >= 0) & (j1 < nx)
        cnt = np.zeros((ny, nx), dtype=np.int32)
        np.add.at(cnt, (i1[keep], j1[keep]), 1)
        return cnt, int((~keep).sum())

    n_cloud, off_c = scatter(movable, np.where(ok_h, h_km, 0.0))
    n_clear, off_k = scatter(clear, hsurf_km)

    out = np.full((ny, nx), np.nan)
    out[n_clear > 0] = 0.0
    out[n_cloud > 0] = 1.0                       # cloud wins a collision

    # a cell that only ever received an uncorrectable cloudy pixel is unknown
    n_lost, _ = scatter(lost, hsurf_km)
    out[(n_lost > 0) & (n_cloud == 0) & (n_clear == 0)] = np.nan

    valid_before = np.isfinite(cma) & in_domain
    valid_after = np.isfinite(out) & in_domain
    diag = dict(
        n_movable=int(movable.sum()), n_lost=int(lost.sum()),
        gap_frac=float(1 - valid_after.sum() / max(valid_before.sum(), 1)),
        collision=int(((n_cloud > 0) & (n_clear > 0) & in_domain).sum()),
        off_grid=off_c + off_k,
        changed=float(np.nanmean((out != cma)[valid_before & valid_after])),
    )
    return out, diag

In [ ]:
# --- demonstration on one slot ----------------------------------------------
demo_t = datetime(2025, 10, 6, 11)
S = read_nwcsaf(nwcsaf_file(demo_t), DOMAIN)
for strict in (False, True):
    cma_corr, diag = parallax_correct(S, HSURF_KM, strict_quality=strict)
    print(f"{demo_t}, strict_quality={strict}:")
    print(f"  {diag['n_movable']} cloudy pixels displaced, {diag['n_lost']} uncorrectable")
    print(f"  cells left unobserved (gaps behind displaced cloud): "
          f"{100 * diag['gap_frac']:.1f}%")
    print(f"  cells receiving both a cloudy and a clear pixel: {diag['collision']}")
    print(f"  cells whose value changed: {100 * diag['changed']:.1f}%")
cma_corr, diag = parallax_correct(S, HSURF_KM)

fields = [(np.where(in_domain, S["cma"], np.nan), "apparent mask (as delivered)"),
          (np.where(in_domain, cma_corr, np.nan), "parallax-corrected"),
          (np.where(in_domain, cma_corr - S["cma"], np.nan), "corrected - apparent")]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6),
                         subplot_kw={"projection": ccrs.PlateCarree()})
for ax, (f_, title) in zip(axes, fields):
    cm, vmin, vmax = ("gray_r", 0, 1) if "-" not in title else ("bwr", -1, 1)
    m = ax.pcolormesh(lon, lat, f_, transform=ccrs.PlateCarree(), cmap=cm,
                      vmin=vmin, vmax=vmax, shading="nearest")
    ax.coastlines("10m")
    ax.add_feature(cfeature.BORDERS, linewidth=0.4)
    ax.set_title(title, fontsize=10)
    fig.colorbar(m, ax=ax, shrink=0.75)
fig.suptitle(f"Parallax correction, {demo_t:%Y-%m-%d %H:%M} UTC "
             f"(white in panel 3 = unchanged, grey = unobserved)")
plt.show()

## 5. Which satellite slot matches an ICON hour

`lff` CLCT is instantaneous and valid exactly on the hour. The satellite slot named `HH:00` is not an
observation at `HH:00` -- it is the start of a sweep that reaches this domain roughly ten minutes later.
Pairing them, as the original notebook does, builds a systematic ten-minute forecast lead into every score,
and cloud fields move enough in ten minutes for that to matter at 3 km.

The acquisition window is read from the files' `start_time`/`end_time`, the position within it from the
scan fraction of section 2, and the slot minimising `|imaged_at - T|` is chosen. The two are compared
directly in section 10, so the effect on `tau*` is measured rather than asserted.

One useful side effect: the two missing slots (2025-10-06 12:00 and 12:15) stop being fatal, because the
hour they would have served is matched to 11:45 anyway.

In [ ]:
# acquisition duration straight from the files
_probe = read_nwcsaf(nwcsaf_file(datetime(2025, 10, 6, 11)), DOMAIN)
_t0 = datetime.fromisoformat(_probe["start_time"])
_t1 = datetime.fromisoformat(_probe["end_time"])
SCAN_DUR_S = (_t1 - _t0).total_seconds()
OFFSET = timedelta(seconds=SCAN_FRAC * SCAN_DUR_S)
print(f"acquisition {_t0:%H:%M:%S} .. {_t1:%H:%M:%S}  ->  {SCAN_DUR_S:.0f} s "
      f"({SCAN_DUR_S / 60:.1f} min)")
print(f"this domain is imaged {OFFSET.total_seconds() / 60:.1f} min after the "
      f"nominal slot time\n")


def choose_slot(t, candidates=(-30, -15, 0, 15)):
    """Nominal slot whose actual imaging time over this domain is closest to t."""
    best, best_err = None, None
    for m in candidates:
        slot = t + timedelta(minutes=m)
        if not os.path.exists(nwcsaf_file(slot)):
            continue
        err = abs((slot + OFFSET - t).total_seconds())
        if best_err is None or err < best_err:
            best, best_err = slot, err
    return best, best_err


print(f"{'ICON hour':<17s} {'naive slot':>11s} {'err':>8s} {'matched':>10s} {'err':>8s}")
for t in [TIMES[0], datetime(2025, 10, 6, 11), datetime(2025, 10, 6, 12), TIMES[-1]]:
    have_naive = os.path.exists(nwcsaf_file(t))
    naive_err = OFFSET.total_seconds() / 60 if have_naive else float("nan")
    slot, _ = choose_slot(t)
    slot_err = (slot + OFFSET - t).total_seconds() / 60 if slot else float("nan")
    print(f"{t:%Y-%m-%d %H:%M} {(t.strftime('%H:%M') if have_naive else 'MISSING'):>11s} "
          f"{naive_err:+7.1f}m {(slot.strftime('%H:%M') if slot else '--'):>10s} "
          f"{slot_err:+7.1f}m")

## 6. Model-side detectability: what MSG could not have seen

The satellite screening of section 3 handles cloud the retrieval *flags* as marginal. It cannot handle the
other half of the problem: cloud ICON puts in the column that is too optically thin for MSG to have
detected at all. That cloud enters CLCT at full weight and shows up as a false alarm, which pushes `tau*`
upwards for a reason that has nothing to do with the forecast being wrong.

So CLCT is recomputed with optically undetectable layers removed. For each model level the in-cloud
condensate path is `q / max(clc, eps) * dp / g`, converted to an optical depth with
`tau = 150 * LWP + 54.5 * IWP` (the standard `3 W / (2 rho r_eff)` with `r_eff` = 10 um liquid, 30 um ice),
and levels below `TAU_DETECT` are zeroed before the layers are recombined under maximum-random overlap.

**The recombination is validated, and it fails.** The same overlap scheme applied to the *unfiltered*
`CLC` profile should reproduce ICON's own diagnostic `CLCT`, and it does not: bias about -5%, RMSE about
11%, correlation 0.97. ICON-CH1 does not diagnose CLCT with plain maximum-random overlap. Substituting the
reconstruction for CLCT would therefore trade a detectability correction for an overlap error of about the
same size.

So the filter is applied **as a ratio** instead. The two reconstructions -- all layers, and detectable
layers only -- are formed with the same scheme, and only their ratio is used:
`CLCT_detectable = CLCT_native * recon_detectable / recon_all`. Whatever the overlap assumption gets wrong
appears in numerator and denominator alike and cancels to first order, and the question the ratio asks --
"what fraction of this column's cover survives the optical-depth filter?" -- is one the reconstruction can
answer even though it cannot reproduce CLCT outright. The validation is still printed every run, because
the size of that mismatch is what justifies the ratio form.

This cell reads four 80-level fields per timestep and is by far the slowest thing here (roughly 20-30 s a
step, so around an hour for the full period). It caches to `DETECT_CACHE`; set `DETECTABILITY_PASS = False`
to skip it, in which case section 10 reports the model-side row as not run.

In [ ]:
G = 9.80665


def max_random_overlap(clc):
    """Total cover from a layer-cover profile, levels ordered top to bottom."""
    eps = 1e-6
    tot = clc[0].copy()
    prev = clc[0]
    for k in range(1, clc.shape[0]):
        cur = clc[k]
        tot = 1 - (1 - tot) * (1 - np.maximum(cur, prev)) / np.clip(1 - prev, eps, None)
        prev = cur
    return np.clip(tot, 0, 1)


def read_profile(fieldlist, name):
    sel = fieldlist.sel({"parameter.variable": name})
    if len(sel) == 0:
        raise KeyError(name)
    xa = ekd.FieldList.from_fields(list(sel)).to_xarray()     # batched: ~5x faster
    nm = name if name in xa else list(xa.data_vars)[0]
    da = xa[nm]
    lv = "level" if "level" in da.dims else [d for d in da.dims if da.sizes[d] > 1][0]
    arr = np.asarray(da.values, dtype="float64")
    lvals = np.asarray(da[lv].values)
    order = np.argsort(lvals)                                  # level 1 = model top
    return arr.reshape(len(lvals), -1)[order]


def detectable_clct(path, tau_detect=TAU_DETECT):
    """CLCT with optically undetectable layers removed, plus the validation of
    the overlap reconstruction against the model's own CLCT."""
    f = ekd.from_source("file", path).to_fieldlist()
    clc = read_profile(f, "CLC") / 100.0
    qc = read_profile(f, "QC")
    qi = read_profile(f, "QI")
    p = read_profile(f, "P")

    if not np.all(np.diff(p.mean(axis=1)) > 0):
        raise RuntimeError("levels are not ordered top-to-bottom -- check the sort")

    dp = np.empty_like(p)
    dp[1:-1] = (p[2:] - p[:-2]) / 2
    dp[0] = p[1] - p[0]
    dp[-1] = p[-1] - p[-2]

    incloud = np.clip(clc, 1e-3, None)
    lwp = qc / incloud * dp / G
    iwp = qi / incloud * dp / G
    od = 150.0 * lwp + 54.5 * iwp

    native = f.sel({"parameter.variable": "CLCT"})[0].to_xarray()
    native = np.asarray(list(native.data_vars.values())[0].values, dtype="float64").ravel()

    recon_all = max_random_overlap(clc)
    recon_det = max_random_overlap(np.where(od >= tau_detect, clc, 0.0))
    # Applied as a RATIO, not as a replacement. The reconstruction does not
    # reproduce ICON's own CLCT (see the validation below), so using it
    # directly would swap a detectability correction for an overlap error.
    # The ratio asks only "what fraction of the reconstructed cover survives
    # the optical-depth filter?" and applies that to the model's own CLCT, so
    # whatever the overlap scheme gets wrong cancels to first order.
    frac = np.divide(recon_det, recon_all, out=np.ones_like(recon_all),
                     where=recon_all > 1e-6)
    return dict(recon=100 * recon_all, native=native, frac=frac,
                filtered=native * np.clip(frac, 0, 1))


if DETECTABILITY_PASS:
    d = detectable_clct(icon_file(demo_t))
    r, nat, fil = d["recon"], d["native"], d["filtered"]
    ok = np.isfinite(r) & np.isfinite(nat)
    bias = float(np.mean(r[ok] - nat[ok]))
    rmse = float(np.sqrt(np.mean((r[ok] - nat[ok]) ** 2)))
    corr = float(np.corrcoef(r[ok], nat[ok])[0, 1])
    print(f"overlap reconstruction vs ICON's own CLCT, {demo_t}:")
    print(f"  bias {bias:+.2f} %, RMSE {rmse:.2f} %, correlation {corr:.4f}")
    if rmse > 5.0:
        print("  the reconstruction does NOT reproduce CLCT -- ICON-CH1 does not "
              "diagnose CLCT\n  with maximum-random overlap. This is why the "
              "filter is applied as a ratio to the\n  model's own CLCT rather "
              "than by substitution: the overlap error cancels.")
    else:
        print("  reconstruction agrees with CLCT to within 5%")
    print(f"  surviving cover fraction: mean {np.mean(d['frac']):.3f}, "
          f"{100 * np.mean(d['frac'] < 0.99):.1f}% of cells affected")
    print(f"  domain-mean CLCT: native {np.mean(nat):.1f} %, "
          f"detectability-filtered {np.mean(fil):.1f} % "
          f"(removed {np.mean(nat) - np.mean(fil):+.1f} %)")
    print(f"\n  sensitivity to TAU_DETECT (the one free constant here):")
    print(f"  {'tau_detect':>11s} {'mean CLCT':>10s} {'removed':>9s}")
    for _td in (0.05, 0.1, 0.2, 0.5, 1.0, 2.0):
        _f = detectable_clct(icon_file(demo_t), tau_detect=_td)["filtered"]
        print(f"  {_td:11.2f} {np.mean(_f):9.1f} % {np.mean(nat) - np.mean(_f):+8.1f} %")
    print("  A large spread here means the model-side row of the ladder is a "
          "statement about\n  the assumed optical properties as much as about "
          "ICON -- read it accordingly.")
else:
    print("DETECTABILITY_PASS = False -- model-side filter not computed")

## 7. Main loop: from fields to histograms

The threshold sweep, the bootstrap, the cross-validation and every stratum all reduce to counting how many
cloudy and how many clear pixels fall in each CLCT bin. So the loop stores exactly that -- a pair of
`(n_time, 201)` count arrays per configuration and per stratum -- and never keeps the fields. Every J
evaluation afterwards is a sum over precomputed counts, which is what makes a thousand bootstrap replicates
across a dozen configurations cheap rather than an overnight job.

Configurations built in one pass:

| forecast | observation | slot |
|---|---|---|
| CLCT | apparent mask | nominal `T` |
| CLCT | apparent mask | scan-matched |
| CLCT | + `ct` screening | scan-matched |
| CLCT | + parallax | scan-matched |
| detectability-filtered CLCT | + parallax | scan-matched |

Strata (`all`, elevation band, illumination, cloud class) are accumulated alongside, so section 13 needs no
second pass.

In [ ]:
def to_bins(clct):
    """Bin index such that (CLCT >= TAUS[k]) is exactly (bin >= k)."""
    v = np.where(np.isfinite(clct), clct, 0.0)
    return np.clip(np.floor(v * BIN_PER_PCT), 0, NBIN - 1).astype(int)


def hist_pair(clct, obs, valid):
    """Per-bin counts of cloudy and clear pixels."""
    m = valid & np.isfinite(clct) & np.isfinite(obs)
    b = to_bins(clct)[m]
    o = obs[m] == 1
    return (np.bincount(b[o], minlength=NBIN).astype(np.int64),
            np.bincount(b[~o], minlength=NBIN).astype(np.int64))


CONFIGS = ["naive", "scan", "scan+ct", "scan+ct+plx", "scan+ct+plx_strict",
           "scan+ct+plx+det"]
STRATA = ["all"] + [f"elev:{n}" for n in elev_names] + \
         ["illum:day", "illum:night", "illum:twilight",
          "class:opaque", "class:fractional", "class:thin", "class:snow_ice"]

H = {(c, s, k): [] for c in CONFIGS for s in STRATA for k in ("cloud", "clear")}
TIMES_used, slot_used, diag_rows = [], [], []

for t in TIMES:
    try:
        fl_t = ekd.from_source("file", icon_file(t)).to_fieldlist()
        sel = fl_t.sel({"parameter.variable": "CLCT"})
        if len(sel) == 0:
            raise FileNotFoundError("no CLCT field")
        xa_t = sel[0].to_xarray()
        nm = "CLCT" if "CLCT" in xa_t else list(xa_t.data_vars)[0]
        clct = remap(np.asarray(xa_t[nm].values, dtype="float64").ravel(),
                     sub, ti, ci, w, coverage, shape)

        slot, _ = choose_slot(t) if SCAN_TIME_MATCH else (t, 0.0)
        if slot is None:
            raise FileNotFoundError("no satellite slot within reach")
        Ss = read_nwcsaf(nwcsaf_file(slot), DOMAIN)
        Sn = read_nwcsaf(nwcsaf_file(t), DOMAIN) if os.path.exists(nwcsaf_file(t)) else None
    except Exception as e:
        print(f"  skip {t}: {type(e).__name__}: {e}")
        continue

    ct = Ss["ct"]
    screen = ~np.isin(ct, CT_SNOW_ICE + CT_FRACTIONAL + CT_THIN)
    if PARALLAX:
        cma_plx, diag = parallax_correct(Ss, HSURF_KM, strict_quality=False)
        cma_plx_s, _ = parallax_correct(Ss, HSURF_KM, strict_quality=True)
    else:
        cma_plx, cma_plx_s, diag = Ss["cma"], Ss["cma"], {}

    clct_det = None
    if DETECTABILITY_PASS:
        try:
            clct_det = remap(detectable_clct(icon_file(t))["filtered"],
                             sub, ti, ci, w, coverage, shape)
        except Exception as e:
            print(f"  {t}: detectability filter failed ({type(e).__name__})")

    variants = {
        "naive": (clct, Sn["cma"] if Sn is not None else None, in_domain),
        "scan": (clct, Ss["cma"], in_domain),
        "scan+ct": (clct, Ss["cma"], in_domain & screen),
        "scan+ct+plx": (clct, cma_plx, in_domain & screen),
        "scan+ct+plx_strict": (clct, cma_plx_s, in_domain & screen),
        "scan+ct+plx+det": (clct_det, cma_plx, in_domain & screen),
    }

    strat_masks = {"all": np.ones(shape, dtype=bool)}
    for k_, n_ in enumerate(elev_names):
        strat_masks[f"elev:{n_}"] = ELEV_BAND == k_
    for nm_, key in (("day", "day"), ("night", "night"), ("twilight", "twilight")):
        strat_masks[f"illum:{nm_}"] = Ss["cond"].get(key, np.zeros(shape, bool))
    strat_masks["class:opaque"] = np.isin(ct, (5, 6, 7, 8, 9))
    strat_masks["class:fractional"] = np.isin(ct, CT_FRACTIONAL)
    strat_masks["class:thin"] = np.isin(ct, CT_THIN)
    strat_masks["class:snow_ice"] = np.isin(ct, CT_SNOW_ICE)

    for cfg, (fc, ob, base) in variants.items():
        for s_ in STRATA:
            if fc is None or ob is None:
                hc = hk = np.zeros(NBIN, dtype=np.int64)
            else:
                hc, hk = hist_pair(fc, ob, base & strat_masks[s_])
            H[(cfg, s_, "cloud")].append(hc)
            H[(cfg, s_, "clear")].append(hk)

    TIMES_used.append(t)
    slot_used.append(slot)
    diag_rows.append(diag)

H = {k: np.stack(v) for k, v in H.items()}
NT = len(TIMES_used)
print(f"\n{NT}/{len(TIMES)} hours usable, {TIMES_used[0]} .. {TIMES_used[-1]}")
if PARALLAX and diag_rows and diag_rows[0]:
    print(f"parallax, averaged over the period: "
          f"{100 * np.mean([d['gap_frac'] for d in diag_rows]):.1f}% of cells left "
          f"unobserved, {100 * np.mean([d['changed'] for d in diag_rows]):.1f}% "
          f"of values changed")
n_shift = sum(1 for t, s in zip(TIMES_used, slot_used) if s != t)
print(f"scan-time matching moved {n_shift}/{NT} hours off the nominal slot")
if HIST_CACHE:
    np.savez_compressed(HIST_CACHE, **{f"{c}|{s}|{k}": H[(c, s, k)]
                                       for c, s, k in H},
                        times=np.array([t.isoformat() for t in TIMES_used]))
    print(f"histograms cached to {HIST_CACHE}")

## 8. Calibration and evaluation pools

Same split as the original -- daytime from 2025-10-07 on is the evaluation pool, everything else is
available for fitting -- with one change: the day/night cut is the satellite's own illumination flag rather
than a recomputed solar zenith angle, and **twilight is excluded from the evaluation pool entirely** rather
than being assigned to whichever side of a 90 degree cut it falls on. Twilight is where a cloud mask is
least reliable, and it is a small fraction of the sample, so it is not worth contaminating the headline
number with.

The hourly steps are not independent. The decorrelation time is measured from the autocorrelation of the
domain-mean bias and drives both the effective sample size and the block length of the bootstrap in
section 11.

In [ ]:
frac_day = np.array([H[("scan", "illum:day", "cloud")][i].sum()
                     + H[("scan", "illum:day", "clear")][i].sum() for i in range(NT)])
frac_twi = np.array([H[("scan", "illum:twilight", "cloud")][i].sum()
                     + H[("scan", "illum:twilight", "clear")][i].sum() for i in range(NT)])
tot = np.array([H[("scan", "all", "cloud")][i].sum()
                + H[("scan", "all", "clear")][i].sum() for i in range(NT)])

is_day = frac_day / np.maximum(tot, 1) > 0.5
is_twi = frac_twi / np.maximum(tot, 1) > 0.5
is_late = np.array([t >= SPLIT_DATE for t in TIMES_used])

EVAL = is_day & is_late & ~is_twi
CALIB = ~EVAL

# --- decorrelation time from the domain-mean bias ---------------------------
def domain_mean_bias(cfg="scan+ct+plx"):
    hc, hk = H[(cfg, "all", "cloud")], H[(cfg, "all", "clear")]
    n = hc.sum(axis=1) + hk.sum(axis=1)
    mean_clct = (hc + hk) @ (TAUS / 100.0) / np.maximum(n, 1)
    obs = hc.sum(axis=1) / np.maximum(n, 1)
    return mean_clct - obs


bias_ts = domain_mean_bias()
bias_ts = bias_ts - np.mean(bias_ts)
n_lags = min(25, NT - 1)
acf = np.array([1.0 if k == 0 else np.corrcoef(bias_ts[:-k], bias_ts[k:])[0, 1]
                for k in range(n_lags)])
DECORR_H = int(next((k for k, a in enumerate(acf) if a < 1 / np.e), len(acf)))
DECORR_H = max(DECORR_H, 1)

fig, ax = plt.subplots(figsize=(6, 2.8))
ax.stem(range(len(acf)), acf)
ax.axhline(1 / np.e, color="k", ls="--", label="1/e")
ax.axvline(DECORR_H, color="r", ls=":", label=f"{DECORR_H} h")
ax.set_xlabel("lag (hours)")
ax.set_ylabel("ACF of domain-mean bias")
ax.legend()
plt.show()

print(f"illumination from the product: {int(is_day.sum())} day, "
      f"{int(is_twi.sum())} twilight, {int((~is_day & ~is_twi).sum())} night")
print(f"pools: calibration {int(CALIB.sum())}, evaluation {int(EVAL.sum())} hours")
print(f"measured decorrelation time: {DECORR_H} h")
for label, sel in (("calibration", CALIB), ("evaluation", EVAL)):
    eff = int(sel.sum()) / DECORR_H
    warn = ("  ** below "
            f"{MIN_EFFECTIVE_N} independent samples **" if eff < MIN_EFFECTIVE_N else "")
    print(f"  {label:12s}: {int(sel.sum()):4d} hours -> effective N ~ {eff:.0f}{warn}")

## 9. Youden's J and `tau*`

`J(tau) = POD(tau) - POFD(tau)`, maximised over the threshold. Peirce (1884) in its meteorological form,
Youden (1950) in general. Because it subtracts the false-alarm rate rather than dividing by a total, J is
insensitive to the base rate -- which matters here, where cloud covers something like 55% of pixels and a
score like the hit rate would be dominated by the climatology.

Evaluated straight from the histograms: `hits(tau)` is the reverse cumulative sum of the cloudy counts
from the bin upwards, `false alarms(tau)` the same over the clear counts. This is **exact**, not an
approximation of a direct contingency table -- bins are `floor(CLCT * 10)` and the threshold is
lower-inclusive, so `CLCT >= TAUS[k]` and `bin >= k` select identically the same pixels. The cell asserts
that identity against a brute-force count and raises if it ever drifts, because the entire notebook
downstream is built on it.

Reported alongside the point value: the **plateau**, the range of thresholds whose J is within one
bootstrap standard error of the maximum. A flat plateau means `tau*` is barely identified by the data, and
the point value should not be quoted to the nearest percent.

In [ ]:
def j_curve(hc, hk):
    """POD, POFD and J at every threshold, from summed histograms."""
    hc = np.asarray(hc, dtype="float64")
    hk = np.asarray(hk, dtype="float64")
    # lower-inclusive: pixels with bin >= k, i.e. CLCT >= TAUS[k]
    above_c = np.cumsum(hc[::-1])[::-1]
    above_k = np.cumsum(hk[::-1])[::-1]
    n_c, n_k = hc.sum(), hk.sum()
    pod = above_c / n_c if n_c > 0 else np.full(NBIN, np.nan)
    pofd = above_k / n_k if n_k > 0 else np.full(NBIN, np.nan)
    return pod, pofd, pod - pofd


def pooled(cfg, stratum, sel):
    return (H[(cfg, stratum, "cloud")][sel].sum(axis=0),
            H[(cfg, stratum, "clear")][sel].sum(axis=0))


def fit_tau(cfg, sel, stratum="all"):
    hc, hk = pooled(cfg, stratum, sel)
    _, _, J = j_curve(hc, hk)
    if not np.isfinite(J).any():
        return np.nan, np.nan, J
    k = int(np.nanargmax(J))
    return TAUS[k], float(J[k]), J


# --- the histogram identity, asserted rather than assumed -------------------
_rng = np.random.default_rng(0)
_c = _rng.uniform(0, 100, 200_000)
_o = (_rng.uniform(0, 1, 200_000) < (_c / 120 + 0.1)).astype(float)
_hc, _hk = hist_pair(_c, _o, np.ones(_c.size, bool))
_pod, _pofd, _ = j_curve(_hc, _hk)
_worst = 0.0
for _k in (0, 137, 500, 1000):
    _p, _ob = _c >= TAUS[_k], _o == 1
    _h, _m = (_p & _ob).sum(), (~_p & _ob).sum()
    _f, _n = (_p & ~_ob).sum(), (~_p & ~_ob).sum()
    _worst = max(_worst, abs(_h / (_h + _m) - _pod[_k]),
                 abs(_f / (_f + _n) - _pofd[_k]))
if _worst > 1e-12:
    raise RuntimeError(f"histogram J disagrees with a direct count by {_worst:.2e}")
print(f"histogram/contingency identity verified (max deviation {_worst:.1e})\n")

# The headline threshold is the fully-corrected one on the *native* CLCT.
# Deliberately NOT the detectability config: that filter changes the forecast
# variable, so its optimum is a threshold on filtered CLCT and cannot be used to
# binarise the CLCT that FSS and SAL actually consume. It stays in the ladder of
# section 10 as a diagnostic.
BASE = "scan+ct+plx"
tau_star, j_calib, J_calib = fit_tau(BASE, CALIB)
_, _, J_eval = fit_tau(BASE, EVAL)
k_star = int(np.rint(tau_star * BIN_PER_PCT))
pod_e, pofd_e, _ = j_curve(*pooled(BASE, "all", EVAL))
j_eval = float(J_eval[k_star])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(TAUS, J_calib, label=f"calibration (n={int(CALIB.sum())} h)")
axes[0].plot(TAUS, J_eval, label=f"evaluation (n={int(EVAL.sum())} h)")
axes[0].axvline(tau_star, color="k", ls="--", label=f"tau* = {tau_star:.1f}%")
axes[0].set_xlabel("CLCT threshold tau (%)")
axes[0].set_ylabel("J = POD - POFD")
axes[0].set_title(f"Threshold selection, exp {EXP}, config '{BASE}'")
axes[0].legend()

axes[1].plot(pofd_e, pod_e, color="C1")
axes[1].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[1].scatter([pofd_e[k_star]], [pod_e[k_star]], color="k", zorder=5,
                label=f"tau* = {tau_star:.1f}%")
axes[1].set_xlabel("POFD")
axes[1].set_ylabel("POD")
axes[1].set_title("ROC, evaluation pool")
axes[1].legend()
plt.show()

print(f"tau* fitted on calibration : {tau_star:.1f}%")
print(f"J(tau*) on calibration     : {j_calib:.3f}")
print(f"J(tau*) on evaluation      : {j_eval:.3f} "
      f"(POD={pod_e[k_star]:.3f}, POFD={pofd_e[k_star]:.3f})")
k_best = int(np.nanargmax(J_eval))
print(f"best achievable on evaluation: J={np.nanmax(J_eval):.3f} at "
      f"tau={TAUS[k_best]:.1f}% -- the {np.nanmax(J_eval) - j_eval:.4f} gap is the "
      f"cost of not fitting in-sample")

## 10. The point of the notebook: how each correction moves `tau*`

The same fit repeated with the corrections switched on one at a time, cumulatively. Each row adds one
thing to the row above, so the change in `tau*` and in J is attributable to that one change. The last
column is the sample the row is fitted on -- screening and parallax both *remove* pixels, and a threshold
fitted on a smaller, cleaner sample is not automatically better, so the cost is shown next to the benefit.

What to read here: if `tau*` barely moves across the ladder, the original number was robust and the
corrections were not worth the loss of sample. If it moves by more than the bootstrap interval of section
11, the uncorrected threshold was absorbing a geometric or timing error, and any downstream use of it --
binarising CLCT for FSS or for SAL objects, as `CLCT_binary_verification.ipynb` does -- inherited that
error.

**The last row is different in kind and must not be read as a sixth point on the same axis.** The
detectability filter changes the forecast variable rather than the observation, so its optimum is a
threshold on filtered CLCT. In practice it collapses towards zero: once optically undetectable layers are
stripped, what remains is close to bimodal and "any cloud at all" becomes the best cut. That is a coherent
result, but the number is not on the same scale as the rows above and cannot be used to binarise the native
CLCT that FSS and SAL consume. Its **J** is the comparable quantity. The headline `tau*` therefore comes
from the last native-CLCT row, not from the bottom of the table.

In [ ]:
LADDER = [("naive", "nominal slot, raw mask (the original section 5)"),
          ("scan", "+ scan-time matched slot"),
          ("scan+ct", "+ ct screening (snow/ice, fractional, thin cirrus)"),
          ("scan+ct+plx", "+ parallax correction"),
          ("scan+ct+plx_strict", "    (variant: parallax, strict ctth quality)"),
          ("scan+ct+plx+det", "+ model-side detectability filter")]

rows = []
for cfg, label in LADDER:
    if cfg == "scan+ct+plx+det" and not DETECTABILITY_PASS:
        rows.append((label, np.nan, np.nan, np.nan, 0))
        continue
    tau_c, j_c, _ = fit_tau(cfg, CALIB)
    if not np.isfinite(tau_c):
        rows.append((label, np.nan, np.nan, np.nan, 0))
        continue
    _, _, Je = fit_tau(cfg, EVAL)
    hc, hk = pooled(cfg, "all", EVAL)
    rows.append((label, tau_c, j_c, float(Je[int(np.rint(tau_c * BIN_PER_PCT))]),
                 int(hc.sum() + hk.sum())))

n0 = rows[0][4]
print(f"{'configuration':<52s} {'tau*':>7s} {'J calib':>8s} {'J eval':>8s} "
      f"{'eval pixels':>13s}")
print("-" * 92)
for label, tau_c, j_c, j_e, npx in rows:
    if not np.isfinite(tau_c):
        print(f"{label:<52s} {'not run':>7s}")
        continue
    frac = f"{100 * npx / n0:5.1f}%" if n0 else "   -- "
    print(f"{label:<52s} {tau_c:6.1f}% {j_c:8.3f} {j_e:8.3f} "
          f"{npx:9d} {frac}")

print("\nNote: the detectability row applies its threshold to a FILTERED CLCT, so its\n"
      "tau* is a threshold on a different variable and is not comparable with the rows\n"
      "above -- its J is. Every row above thresholds the native CLCT, which is what\n"
      "downstream products (FSS, SAL objects) binarise, so tau* is taken from the last\n"
      "of those rather than from the bottom of the table.")

fig, ax = plt.subplots(figsize=(7, 4.2))
for cfg, label in LADDER:
    if cfg == "scan+ct+plx+det" and not DETECTABILITY_PASS:
        continue
    _, _, J = fit_tau(cfg, CALIB)
    ax.plot(TAUS, J, label=label.replace("+ ", ""), lw=1.4)
ax.set_xlabel("CLCT threshold tau (%)")
ax.set_ylabel("J on the calibration pool")
ax.set_title(f"Effect of each correction on the J curve, exp {EXP}")
ax.legend(fontsize=8)
plt.show()

## 11. How well is `tau*` actually determined? (moving-block bootstrap)

Resampling pixels would be meaningless -- neighbouring pixels are not independent draws and there are
hundreds of thousands of them, so any pixel-level interval would be absurdly narrow. The block bootstrap
resamples *time*, in contiguous blocks of the measured decorrelation length, which is the scale on which
this data actually carries new information. Because everything is precomputed histograms, a replicate is a
sum of `n/L` rows and a thousand of them cost nothing.

Two numbers come out, and the second is the more important:

- a confidence interval on `tau*` itself, and on `J(tau*)`;
- the **plateau**: every threshold whose calibration J lies within one bootstrap standard error of the
  maximum. If that plateau is 30 points wide, `tau*` is a convention, not a measurement, and quoting it to
  the nearest percent -- as the original notebook does -- reads far more precision into the data than is
  there.

In [ ]:
rng = np.random.default_rng(20251007)


def block_bootstrap(cfg, sel, n_boot=N_BOOT, L=None, stratum="all"):
    L = DECORR_H if L is None else L
    idx = np.where(sel)[0]
    n = idx.size
    if n < L:
        return np.array([]), np.array([])
    hc_all = H[(cfg, stratum, "cloud")][idx]
    hk_all = H[(cfg, stratum, "clear")][idx]
    n_blocks = max(int(np.ceil(n / L)), 1)
    starts_pool = np.arange(0, max(n - L + 1, 1))

    taus_b, js_b = np.empty(n_boot), np.empty(n_boot)
    for b in range(n_boot):
        st = rng.choice(starts_pool, size=n_blocks, replace=True)
        take = np.concatenate([np.arange(s, min(s + L, n)) for s in st])
        _, _, J = j_curve(hc_all[take].sum(axis=0), hk_all[take].sum(axis=0))
        if np.isfinite(J).any():
            k = int(np.nanargmax(J))
            taus_b[b], js_b[b] = TAUS[k], J[k]
        else:
            taus_b[b] = js_b[b] = np.nan
    return taus_b, js_b


taus_b, js_b = block_bootstrap(BASE, CALIB)
tau_lo, tau_hi = np.nanpercentile(taus_b, [2.5, 97.5])
j_lo, j_hi = np.nanpercentile(js_b, [2.5, 97.5])
se_j = float(np.nanstd(js_b))

plateau = TAUS[J_calib >= np.nanmax(J_calib) - se_j]
p_lo, p_hi = float(plateau.min()), float(plateau.max())

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].hist(taus_b[np.isfinite(taus_b)], bins=40, color="C0")
axes[0].axvline(tau_star, color="k", ls="--")
axes[0].set_xlabel("bootstrap tau* (%)")
axes[0].set_ylabel("replicates")
axes[0].set_title(f"tau* = {tau_star:.1f}%, 95% CI [{tau_lo:.1f}, {tau_hi:.1f}]")

axes[1].plot(TAUS, J_calib, color="C0")
axes[1].axhspan(np.nanmax(J_calib) - se_j, np.nanmax(J_calib), color="C1", alpha=0.25)
axes[1].axvspan(p_lo, p_hi, color="C1", alpha=0.15)
axes[1].axvline(tau_star, color="k", ls="--")
axes[1].set_xlabel("CLCT threshold tau (%)")
axes[1].set_ylabel("J (calibration)")
axes[1].set_title(f"plateau within 1 SE: {p_lo:.1f} .. {p_hi:.1f}%")
plt.show()

print(f"block bootstrap, {N_BOOT} replicates, block length {DECORR_H} h "
      f"({int(CALIB.sum())} calibration hours)")
print(f"  tau*    = {tau_star:.1f}%   95% CI [{tau_lo:.1f}, {tau_hi:.1f}]  "
      f"(width {tau_hi - tau_lo:.1f} points)")
print(f"  J(tau*) = {j_calib:.3f}   95% CI [{j_lo:.3f}, {j_hi:.3f}]  SE {se_j:.4f}")
print(f"  plateau within 1 SE of the maximum: {p_lo:.1f} .. {p_hi:.1f}% "
      f"({p_hi - p_lo:.1f} points wide)")
if p_hi - p_lo > 20:
    print("  ** the J curve is flat: tau* is weakly identified and should be "
          "quoted as a range, not a value **")

## 12. k-fold cross-validation over time blocks

The single calibration/evaluation split answers "does this threshold transfer to held-out data once?".
Cross-validation answers the more useful question: **how much does `tau*` move depending on which data it
was fitted on?** If refitting on five of six time blocks gives thresholds spread across 30 points, then the
single split's `tau*` was a draw from a wide distribution and the specific value carries little meaning.

Folds are contiguous blocks of hours, not random subsets -- randomly interleaved folds would put hour `T`
in the training set and hour `T+1` in the test set, which are almost the same field, and would report
optimistic transfer that will not hold for a different week. The whole usable period is used here rather
than the calibration pool alone, so that the folds are contiguous in time and each is large enough to fit
on; the date split of section 8 is what protects the headline number, and this section is a stability
diagnostic rather than a second source of it.

In [ ]:
usable = np.where(np.isfinite(domain_mean_bias(BASE)))[0]
folds = np.array_split(usable, N_FOLDS)

print(f"{'fold':>5s} {'hours':>18s} {'n':>5s} {'tau* (fit on rest)':>20s} "
      f"{'J on fold':>10s} {'J at global tau*':>17s}")
cv_taus, cv_js = [], []
for f_i, fold in enumerate(folds):
    tr = np.ones(NT, dtype=bool)
    tr[fold] = False
    te = np.zeros(NT, dtype=bool)
    te[fold] = True
    tau_f, _, _ = fit_tau(BASE, tr)
    _, _, J_te = fit_tau(BASE, te)
    if not (np.isfinite(tau_f) and np.isfinite(J_te).any()):
        continue
    j_f = float(J_te[int(np.rint(tau_f * BIN_PER_PCT))])
    j_g = float(J_te[k_star])
    cv_taus.append(tau_f)
    cv_js.append(j_f)
    print(f"{f_i:5d} {TIMES_used[fold[0]]:%m-%d %Hh}-{TIMES_used[fold[-1]]:%m-%d %Hh} "
          f"{len(fold):5d} {tau_f:19.1f}% {j_f:10.3f} {j_g:17.3f}")

cv_taus = np.array(cv_taus)
print(f"\ntau* across folds: {cv_taus.min():.1f} .. {cv_taus.max():.1f}%, "
      f"spread {cv_taus.max() - cv_taus.min():.1f} points, "
      f"sd {cv_taus.std():.1f}")
print(f"J across folds:    {np.mean(cv_js):.3f} +/- {np.std(cv_js):.3f}")
if cv_taus.max() - cv_taus.min() > 20:
    print("  ** tau* is not stable across time blocks: it is describing the "
          "weather of the fitting period as much as the model **")

## 13. Where a single threshold does not fit

One `tau*` for the whole domain and the whole period assumes the CLCT-to-cloud-mask relationship is the
same over the Plateau at 400 m and the Alps at 3000 m, and in daylight as at night. Refitting inside each
stratum tests that. It is a diagnostic, not a proposal to use a different threshold in each stratum --
that would fit a dozen parameters on a week of data. What it is for: if `tau*` differs sharply between
strata, the single global threshold is a compromise that is wrong nearly everywhere, and downstream
products built on it (binarised FSS, SAL objects) carry a spatially-structured error rather than a uniform
one. Elevation is the one to watch, because it is where the parallax displacement and the snow ambiguity
both bite hardest.

**Cloud class is handled separately, and not by refitting.** A stratum defined by the observed cloud type
*determines the observation*: `class:opaque` is about 99.5% cloudy, `class:snow_ice` almost entirely clear.
There is no meaningful mix of events and non-events inside such a stratum, so POD and POFD cannot both be
formed and a threshold fitted there would be circular by construction -- the earlier version of this
notebook printed "too few pixels to fit" for every class, which is that circularity showing up as a
symptom rather than being diagnosed.

The non-circular question is instead: **at the global `tau*`, how often does ICON put cloud where NWCSAF
sees each class of cloud?** That is a detection rate per class, fitted nowhere, and it is the more useful
number anyway -- it says which cloud types the model misses. The cloud-free classes get the complementary
quantity, the false-alarm rate. Both come from the unscreened configuration, since the `ct` screening
removes several of these classes from `BASE` by design.

In [ ]:
# --- (a) strata where a threshold can legitimately be fitted -----------------
FITTABLE = ["all"] + [f"elev:{n}" for n in elev_names] + \
           ["illum:day", "illum:night", "illum:twilight"]

print(f"{'stratum':<28s} {'tau* calib':>11s} {'J calib':>8s} {'J eval':>8s} "
      f"{'cloud frac':>11s} {'pixels':>12s}")
print("-" * 84)
strat_out = {}
for s_ in FITTABLE:
    hc_c, hk_c = pooled(BASE, s_, CALIB)
    if hc_c.sum() < 5000 or hk_c.sum() < 5000:
        print(f"{s_:<28s} {'too few pixels to fit':>11s}")
        continue
    _, _, Jc = j_curve(hc_c, hk_c)
    k_s = int(np.nanargmax(Jc))
    hc_e, hk_e = pooled(BASE, s_, EVAL)
    _, _, Je = j_curve(hc_e, hk_e)
    j_e_s = float(Je[k_s]) if np.isfinite(Je).any() else np.nan
    cf = hc_c.sum() / (hc_c.sum() + hk_c.sum())
    strat_out[s_] = (TAUS[k_s], float(Jc[k_s]), j_e_s)
    je_txt = f"{j_e_s:8.3f}" if np.isfinite(j_e_s) else f"{'--':>8s}"
    print(f"{s_:<28s} {TAUS[k_s]:10.1f}% {Jc[k_s]:8.3f} {je_txt} "
          f"{cf:11.3f} {int(hc_c.sum() + hk_c.sum()):12d}")

elev_taus = [strat_out[f"elev:{n}"][0] for n in elev_names if f"elev:{n}" in strat_out]
if len(elev_taus) > 1:
    spread = max(elev_taus) - min(elev_taus)
    print(f"\ntau* across elevation bands: {min(elev_taus):.1f} .. {max(elev_taus):.1f}% "
          f"(spread {spread:.1f} points; compare with the {tau_hi - tau_lo:.1f}-point "
          f"bootstrap CI on the global value)")
    if spread > tau_hi - tau_lo:
        print("  ** the elevation spread exceeds the sampling uncertainty on the "
              "global tau*: one\n     threshold for the whole domain is a compromise, "
              "and the error it leaves behind\n     is terrain-structured rather than "
              "uniform **")

# --- (b) cloud class: detection rate at the global tau*, not a refit ---------
# A class stratum determines the observation, so no threshold can be fitted in
# it. What is well defined is how often the model exceeds the global tau* on
# pixels of each observed class: a hit rate for the cloudy classes, a
# false-alarm rate for the cloud-free ones. Taken from the unscreened config,
# since the ct screening removes several of these classes from BASE.
print(f"\n\nDetection of each observed cloud class at the global tau* = "
      f"{tau_star:.1f}%  (config 'scan', unscreened)")
print(f"{'observed class':<28s} {'kind':>10s} {'n pixels':>11s} "
      f"{'CLCT >= tau*':>13s}")
print("-" * 66)
for s_ in [s for s in STRATA if s.startswith("class:")]:
    hc_c, hk_c = pooled("scan", s_, CALIB)
    n_c, n_k = hc_c.sum(), hk_c.sum()
    if n_c + n_k < 5000:
        print(f"{s_:<28s} {'--':>10s} {int(n_c + n_k):11d}   too few pixels")
        continue
    # the class decides which side it belongs on; use whichever is populated
    if n_c >= n_k:
        rate = j_curve(hc_c, hk_c)[0][k_star]      # POD
        kind, n_ = "cloudy", n_c
    else:
        rate = j_curve(hc_c, hk_c)[1][k_star]      # POFD
        kind, n_ = "clear", n_k
    print(f"{s_:<28s} {kind:>10s} {int(n_):11d} {rate:12.3f}")
print("\nFor the cloudy classes this is a hit rate (higher is better); for the "
      "clear ones a\nfalse-alarm rate (lower is better). A low hit rate on "
      "'thin' or 'fractional' is the\nsignature of cloud the model has but the "
      "satellite can barely see -- the same effect\nthe model-side filter in "
      "section 6 attacks from the other direction.")

## 14. Summary

In [ ]:
print("=" * 88)
print(f"exp {EXP}   {TIMES_used[0]:%Y-%m-%d %H} .. {TIMES_used[-1]:%Y-%m-%d %H} UTC   "
      f"{NT} hours, grid {shape}, {int(in_domain.sum())} cells in domain")
print(f"config: '{BASE}'  (parallax={PARALLAX}, scan-time match={SCAN_TIME_MATCH}, "
      f"detectability={DETECTABILITY_PASS})")
print(f"pools: calibration {int(CALIB.sum())} h (fitting only), evaluation "
      f"{int(EVAL.sum())} h (daytime from {SPLIT_DATE:%Y-%m-%d}, twilight excluded)")
print(f"effective sample size of the evaluation pool: ~{int(EVAL.sum()) / DECORR_H:.0f} "
      f"({DECORR_H} h decorrelation)")
print("-" * 88)
print(f"tau*                    {tau_star:.1f}%   95% CI [{tau_lo:.1f}, {tau_hi:.1f}]   "
      f"plateau {p_lo:.1f}-{p_hi:.1f}%")
print(f"tau* across CV folds    {cv_taus.min():.1f} .. {cv_taus.max():.1f}% "
      f"(sd {cv_taus.std():.1f})")
print(f"J(tau*) on evaluation   {j_eval:.3f}   POD {pod_e[k_star]:.3f}, "
      f"POFD {pofd_e[k_star]:.3f}")
print(f"in-sample gap           {np.nanmax(J_eval) - j_eval:.4f}")
print("-" * 88)
print("movement of tau* along the correction ladder "
      "(last row thresholds a different variable):")
for label, tau_c, j_c, j_e, npx in rows:
    if np.isfinite(tau_c):
        print(f"  {label:<50s} {tau_c:6.1f}%  (J eval {j_e:.3f})")
    else:
        print(f"  {label:<50s} {'not run':>7s}")
print("=" * 88)
if tau_hi - tau_lo > 10 or (len(cv_taus) and cv_taus.max() - cv_taus.min() > 20):
    print("READ AS A RANGE, NOT A VALUE: the bootstrap interval and/or the fold "
          "spread are wide\nrelative to the movement along the ladder. Quote "
          "tau* to the nearest 5% at best.")

## Notes

**What is fitted where.** `tau*` is fitted on the calibration pool only, and every number reported on the
evaluation pool comes from that fit. The detectability threshold `TAU_DETECT`, the elevation band edges and
the bootstrap block length are fixed a priori or measured from the data rather than tuned against J;
`DECORR_H` is measured from the autocorrelation in section 8 and then used, not chosen.

**Corrections, and what each is really doing.**

- *Parallax* is applied to the observation, using the `ctth_alti` retrieval in the same file. It is a
  genuine correction for cloudy pixels with a height, an approximation for clear pixels (terrain height
  stands in for the surface), and nothing at all for the ~9% of cloudy pixels with no reliable retrieval,
  which are dropped. The gaps it opens behind displaced cloud are left as missing rather than filled.
- *Quality screening* could not be done the intended way: these exports carry no `cma_quality` or
  `cma_conditions`, so the cloud mask itself arrives with no confidence information. The `ctth_*` flags
  gate the height retrieval and hence the parallax correction; screening of the mask goes through the `ct`
  cloud-type classes instead. This is a weaker instrument than a real CMA confidence field and it is worth
  saying so. If the mask's own flags matter, they would have to be re-extracted from NWC SAF with the CMA
  ancillary variables retained.
- *Scan timing* removes a systematic ~10 minute offset, not a random one. It is the cheapest correction
  here and the only one that costs no sample.
- *Detectability* is one-sided by construction: it removes model cloud MSG could not have seen, but nothing
  puts back satellite cloud ICON could not represent. The `ct`-based screening handles part of the other
  direction. The optical-depth constants (`r_eff` 10 um liquid, 30 um ice) are conventional values, not
  ICON's own microphysical assumptions, so the filter is a plausibility filter rather than a radiative
  transfer calculation. It is also applied as a ratio because the overlap reconstruction it rests on does
  not reproduce ICON's CLCT -- that cancels the overlap error but does not make the layer-by-layer optical
  depths any more authoritative. A real forward operator (RTTOV through the model column) is the honest
  version of this step and is out of scope.
- *Parallax* is not gated on `ctth_quality` by default. 28% of pixels carry a `bad` height flag, and
  strict gating leaves about a third of the domain unobserved; the ladder carries both so the choice is
  visible. Neither variant corrects the ~9% of cloudy pixels with no height retrieval at all.

**Known limitations.**
- The evaluation pool is a few days of daytime hours; the effective sample size is small enough that the
  differences along the correction ladder should be compared against the bootstrap interval before being
  believed, and 801-vs-802 differences almost certainly cannot be resolved at this sample size.
- The threshold grid is 0.1% and the histogram evaluation is exact rather than approximate, so threshold
  resolution is nowhere near the binding constraint -- the plateau width is.
- The `ct` screening removes fractional and thin-cirrus pixels from the fit. Those are real cloud, so the
  screened `tau*` answers "what threshold best separates cloud MSG can see confidently?" -- a slightly
  different question from the unscreened one. The ladder in section 10 is what makes that visible.
- One `tau*` is fitted for the whole period. Section 13 shows whether that is defensible; it does not fix
  it.

**Using `tau*` downstream.** `CLCT_binary_verification.ipynb` uses its `tau*` to binarise CLCT for the FSS
of section 6 and for the SAL objects of section 8. If the ladder here moves the threshold appreciably, both
of those should be re-run with the corrected value -- and given the plateau width, ideally re-run at the
plateau edges too, so that the sensitivity of the FSS and SAL conclusions to the threshold is visible
rather than assumed away.